# Transfermarkt Bundesliga 2026/27 Squad Scraper

This notebook retrieves the 2026/27 squad shown on each of the 18 configured Transfermarkt club pages. All webpage retrieval is performed with one reusable undetected Chrome session; the notebook does not use requests.

## Website and extraction target

- The notebook accesses only the configured squad pages on **https://www.transfermarkt.com**.
- It parses the main Transfermarkt **table.items** squad table and only its direct outer player rows, not the separate squad-summary table or the nested layout rows inside player cells.
- It extracts team and player IDs, shirt number, player name and profile URL, position, age, all listed nationalities, contract expiration, displayed market value, and a numeric EUR market value.

## Portable execution and outputs

- No machine-specific paths are embedded. Squad exports are written to `outputs/transfermarkt/squads`, while diagnostic HTML is written to `outputs/transfermarkt/debug`.
- Paths are resolved through `project_paths.py`; start Jupyter from the project root or open the notebook from within the project tree.
- A run produces timestamped **JSON** and **CSV** files. JSON retains nationality arrays; CSV joins them with a pipe. If a squad table cannot be loaded, the received HTML is saved as **debug_team-name.html** for inspection.

## Required packages

Install Python, Jupyter, pandas, beautifulsoup4, undetected-chromedriver, and Selenium. Chrome is started exactly with **uc.Chrome(version_main=150)**, so a compatible Chrome 150 installation is required.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    TRANSFERMARKT_DEBUG_DIR,
    TRANSFERMARKT_SQUADS_DIR,
    ensure_directory,
)


## 1. Imports and configuration

The configuration contains every requested Transfermarkt URL, timeout and delay settings, the output schema, and the optional portable directory override.


In [2]:
# Import the libraries required by this notebook step.
import json
import math
import random
import re
import time
from datetime import datetime
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from numbers import Integral, Real
from pathlib import Path
from typing import Any
from urllib.parse import urljoin

# Handle expected failures with a clear, actionable message.
try:
    import pandas as pd
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from IPython.display import display
    from selenium.common.exceptions import TimeoutException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Required packages are missing from this notebook kernel. Install them with: "
        "%pip install pandas beautifulsoup4 undetected-chromedriver selenium"
    ) from exc

# Set workflow configuration value: SEASON_ID.
SEASON_ID = 2026
# Set workflow configuration value: TRANSFERMARKT_BASE_URL.
TRANSFERMARKT_BASE_URL = "https://www.transfermarkt.com"
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 45
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 30
# Set workflow configuration value: MIN_DELAY_SECONDS.
MIN_DELAY_SECONDS = 1.5
# Set workflow configuration value: MAX_DELAY_SECONDS.
MAX_DELAY_SECONDS = 3.0

# Set workflow configuration value: TEAM_URLS.
TEAM_URLS = {
    "Bayern Munich": "https://www.transfermarkt.com/fc-bayern-munchen/kader/verein/27/saison_id/2026",
    "RB Leipzig": "https://www.transfermarkt.com/rasenballsport-leipzig/kader/verein/23826/saison_id/2026",
    "Bayer Leverkusen": "https://www.transfermarkt.com/bayer-04-leverkusen/kader/verein/15/saison_id/2026",
    "Borussia Dortmund": "https://www.transfermarkt.com/borussia-dortmund/kader/verein/16/saison_id/2026",
    "VfB Stuttgart": "https://www.transfermarkt.com/vfb-stuttgart/kader/verein/79/saison_id/2026",
    "Eintracht Frankfurt": "https://www.transfermarkt.com/eintracht-frankfurt/kader/verein/24/saison_id/2026",
    "TSG Hoffenheim": "https://www.transfermarkt.com/tsg-1899-hoffenheim/kader/verein/533/saison_id/2026",
    "SC Freiburg": "https://www.transfermarkt.com/sc-freiburg/kader/verein/60/saison_id/2026",
    "Mainz 05": "https://www.transfermarkt.com/1-fsv-mainz-05/kader/verein/39/saison_id/2026",
    "FC Augsburg": "https://www.transfermarkt.com/fc-augsburg/kader/verein/167/saison_id/2026",
    "Werder Bremen": "https://www.transfermarkt.com/sv-werder-bremen/kader/verein/86/saison_id/2026",
    "Borussia Mönchengladbach": "https://www.transfermarkt.com/borussia-monchengladbach/kader/verein/18/saison_id/2026",
    "1. FC Köln": "https://www.transfermarkt.com/1-fc-koln/kader/verein/3/saison_id/2026",
    "Union Berlin": "https://www.transfermarkt.com/1-fc-union-berlin/kader/verein/89/saison_id/2026",
    "Hamburger SV": "https://www.transfermarkt.com/hamburger-sv/kader/verein/41/saison_id/2026",
    "FC Schalke 04": "https://www.transfermarkt.com/fc-schalke-04/kader/verein/33/saison_id/2026",
    "SV Elversberg": "https://www.transfermarkt.com/sv-07-elversberg/kader/verein/64/saison_id/2026",
    "SC Paderborn": "https://www.transfermarkt.com/sc-paderborn-07/kader/verein/127/saison_id/2026",
}

# Set workflow configuration value: PLAYER_COLUMNS.
PLAYER_COLUMNS = [
    "team",
    "team_id",
    "season_id",
    "shirt_number",
    "player_name",
    "player_id",
    "position",
    "age",
    "nationalities",
    "contract_until",
    "market_value",
    "market_value_eur",
    "player_url",
]

# Set workflow configuration value: SUMMARY_COLUMNS.
SUMMARY_COLUMNS = [
    "team",
    "players_scraped",
    "players_with_market_value",
    "total_market_value_eur",
    "average_market_value_eur",
    "median_market_value_eur",
    "highest_market_value_eur",
]


## 2. Resolve the portable runtime directory

Every generated path derives from one runtime directory. This keeps the notebook movable and avoids any dependency on the directory where it was created.


In [3]:
# Run this self-contained workflow step using the prepared inputs.
base_directory = ensure_directory(TRANSFERMARKT_SQUADS_DIR)
print(f"Squad output directory: {base_directory}")


Squad output directory: C:\kickbase project\outputs\transfermarkt\squads


## 3. Parsing and validation helpers

These helpers normalize text, extract IDs, convert displayed market values, map table headers to direct cells, and parse one outer player row defensively.


In [4]:
# Clean text for reuse in the workflow.
def clean_text(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).replace("\xa0", " ")
    cleaned = re.sub(r"\s+", " ", text).strip()
    return cleaned or None


# Extract team ID for reuse in the workflow.
def extract_team_id(url: str) -> int | None:
    match = re.search(r"/verein/(\d+)(?:/|$)", url)
    return int(match.group(1)) if match else None


# Extract player ID for reuse in the workflow.
def extract_player_id(player_url: str | None) -> int | None:
    if not player_url:
        return None
    match = re.search(r"/spieler/(\d+)(?:[/?#]|$)", player_url)
    return int(match.group(1)) if match else None


# Parse and validate optional integer for reuse in the workflow.
def parse_optional_integer(value: Any) -> int | None:
    cleaned = clean_text(value)
    if cleaned and re.fullmatch(r"\d+", cleaned):
        return int(cleaned)
    return None


# Parse and validate market value for reuse in the workflow.
def parse_market_value(value: Any) -> int | None:
    cleaned = clean_text(value)
    if cleaned is None or cleaned.casefold() in {"-", "—", "n/a", "none", "null"}:
        return None

    normalized = cleaned.replace("€", "").replace("EUR", "").strip()
    match = re.fullmatch(r"([+-]?[\d.,]+)\s*(bn|m|k)?", normalized, flags=re.IGNORECASE)
    if not match:
        return None

    number_text = match.group(1)
    suffix = (match.group(2) or "").casefold()

    # Choose the appropriate path for the current data state.
    if "," in number_text and "." in number_text:
        number_text = number_text.replace(",", "")
    elif "," in number_text:
        comma_groups = number_text.lstrip("+-").split(",")
        # Choose the appropriate path for the current data state.
        if not suffix and len(comma_groups) > 1 and all(
            len(group) == 3 for group in comma_groups[1:]
        ):
            number_text = number_text.replace(",", "")
        else:
            number_text = number_text.replace(",", ".")

    multipliers = {
        "": Decimal("1"),
        "k": Decimal("1000"),
        "m": Decimal("1000000"),
        "bn": Decimal("1000000000"),
    }
    # Handle expected failures with a clear, actionable message.
    try:
        euro_value = Decimal(number_text) * multipliers[suffix]
    except (InvalidOperation, KeyError):
        return None

    return int(euro_value.quantize(Decimal("1"), rounding=ROUND_HALF_UP))


# Extract squad size for reuse in the workflow.
def extract_squad_size(soup: Any) -> int | None:
    details = soup.select_one(".data-header__details, .data-header__info-box")
    search_text = clean_text(details.get_text(" ", strip=True)) if details else None
    if not search_text:
        search_text = clean_text(soup.get_text(" ", strip=True))
    match = re.search(r"\bSquad size:\s*(\d+)\b", search_text or "", flags=re.IGNORECASE)
    return int(match.group(1)) if match else None


# Normalize header for reuse in the workflow.
def normalize_header(value: Any) -> str:
    return re.sub(r"[^a-z0-9#]+", "", (clean_text(value) or "").casefold())


# Build column map for reuse in the workflow.
def build_column_map(table: Any) -> dict[str, int]:
    column_map: dict[str, int] = {}
    # Process each available item while preserving the current workflow state.
    for index, header in enumerate(table.select("thead th")):
        normalized = normalize_header(header.get_text(" ", strip=True))
        if normalized:
            column_map[normalized] = index
    return column_map


# Handle mapped cell for reuse in the workflow.
def get_mapped_cell(
    direct_cells: list[Any], column_map: dict[str, int], *header_names: str
) -> Any | None:
    # Process each available item while preserving the current workflow state.
    for header_name in header_names:
        index = column_map.get(normalize_header(header_name))
        if index is not None and index < len(direct_cells):
            return direct_cells[index]
    return None


# Handle debug filename for reuse in the workflow.
def make_debug_filename(team_name: str) -> str:
    safe_team_name = re.sub(r"[^\w.-]+", "_", team_name, flags=re.UNICODE).strip("_.")
    return f"debug_{safe_team_name or 'unknown_team'}.html"


# Save debug html for reuse in the workflow.
def save_debug_html(team_name: str, html: str) -> Path:
    debug_path = ensure_directory(TRANSFERMARKT_DEBUG_DIR) / make_debug_filename(team_name)
    debug_path.write_text(html, encoding="utf-8")
    return debug_path


# Parse and validate player row for reuse in the workflow.
def parse_player_row(
    row: Any,
    team_name: str,
    team_url: str,
    column_map: dict[str, int],
    validation_warnings: list[str],
) -> dict[str, Any]:
    direct_cells = row.find_all("td", recursive=False)
    profile_link = row.select_one("a[href*='/profil/spieler/']")

    player_name = clean_text(profile_link.get_text(" ", strip=True)) if profile_link else None
    if not player_name:
        portrait = row.select_one("table.inline-table img[alt], table.inline-table img[title]")
        if portrait:
            player_name = clean_text(portrait.get("alt") or portrait.get("title"))

    relative_player_url = profile_link.get("href") if profile_link else None
    player_url = urljoin(TRANSFERMARKT_BASE_URL, relative_player_url) if relative_player_url else None

    shirt_cell = row.select_one("td.rueckennummer")
    shirt_number = parse_optional_integer(
        shirt_cell.get_text(" ", strip=True) if shirt_cell else None
    )

    position = None
    inline_table = row.select_one("td.posrela table.inline-table")
    if inline_table:
        inline_rows = inline_table.find_all("tr")
        if len(inline_rows) > 1:
            position_cell = inline_rows[1].find("td")
            position = clean_text(
                position_cell.get_text(" ", strip=True) if position_cell else None
            )

    age_cell = get_mapped_cell(direct_cells, column_map, "Age")
    if age_cell is None:
        # Process each available item while preserving the current workflow state.
        for candidate_cell in direct_cells:
            if candidate_cell is shirt_cell:
                continue
            candidate_text = clean_text(candidate_cell.get_text(" ", strip=True))
            if candidate_text and re.fullmatch(r"\d{1,2}", candidate_text):
                age_cell = candidate_cell
                break
    age = parse_optional_integer(age_cell.get_text(" ", strip=True) if age_cell else None)

    nationality_cell = get_mapped_cell(
        direct_cells, column_map, "Nat.", "Nationality", "Nationalities"
    )
    flag_elements = (
        nationality_cell.select("img.flaggenrahmen")
        if nationality_cell is not None
        else row.select("img.flaggenrahmen")
    )
    nationalities: list[str] = []
    # Process each available item while preserving the current workflow state.
    for flag in flag_elements:
        nationality = clean_text(flag.get("title") or flag.get("alt"))
        if nationality and nationality not in nationalities:
            nationalities.append(nationality)

    contract_cell = get_mapped_cell(
        direct_cells, column_map, "Contract", "Contract until"
    )
    if contract_cell is None:
        # Process each available item while preserving the current workflow state.
        for candidate_cell in direct_cells:
            candidate_text = clean_text(candidate_cell.get_text(" ", strip=True))
            if candidate_text and re.fullmatch(r"\d{2}/\d{2}/\d{4}", candidate_text):
                contract_cell = candidate_cell
                break
    contract_until = clean_text(
        contract_cell.get_text(" ", strip=True) if contract_cell else None
    )
    if contract_until and contract_until.casefold() in {"-", "—", "n/a", "none", "null"}:
        contract_until = None

    market_cell = get_mapped_cell(
        direct_cells, column_map, "Market value", "Market"
    )
    if market_cell is None:
        market_link = row.select_one("a[href*='/marktwertverlauf/spieler/']")
        market_cell = market_link.find_parent("td") if market_link else None

    if market_cell is None:
        simplified_row_text = clean_text(row.get_text(" ", strip=True)) or "<empty row>"
        warning = (
            f"WARNING: {team_name} | {player_name or '<unknown player>'} | "
            f"market-value element not found | row: {simplified_row_text}"
        )
        validation_warnings.append(warning)
        print(warning)

    market_value = clean_text(
        market_cell.get_text(" ", strip=True) if market_cell else None
    )

    return {
        "team": team_name,
        "team_id": extract_team_id(team_url),
        "season_id": SEASON_ID,
        "shirt_number": shirt_number,
        "player_name": player_name,
        "player_id": extract_player_id(player_url),
        "position": position,
        "age": age,
        "nationalities": nationalities,
        "contract_until": contract_until,
        "market_value": market_value,
        "market_value_eur": parse_market_value(market_value),
        "player_url": player_url,
    }


# Handle team for reuse in the workflow.
def scrape_team(
    driver: Any,
    team_name: str,
    url: str,
    validation_warnings: list[str],
) -> tuple[list[dict[str, Any]], int | None]:
    wait = WebDriverWait(driver, WAIT_TIMEOUT_SECONDS)

    # Handle expected failures with a clear, actionable message.
    try:
        driver.get(url)
        wait.until(
            lambda current_driver: current_driver.execute_script(
                "return document.readyState"
            ) == "complete"
        )
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.items")))
    except TimeoutException as exc:
        html = driver.page_source
        debug_path = save_debug_html(team_name, html)
        raise RuntimeError(
            f"Timed out waiting for table.items. Debug HTML saved to {debug_path}."
        ) from exc

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", class_="items")
    # Validate the input before continuing with later processing.
    if table is None:
        debug_path = save_debug_html(team_name, html)
        raise RuntimeError(
            f"No table.items squad table found. Debug HTML saved to {debug_path}."
        )

    outer_tbody = table.find("tbody", recursive=False)
    # Validate the input before continuing with later processing.
    if outer_tbody is None:
        debug_path = save_debug_html(team_name, html)
        raise RuntimeError(
            f"The squad table has no outer tbody. Debug HTML saved to {debug_path}."
        )

    column_map = build_column_map(table)
    player_records: list[dict[str, Any]] = []
    # Process each available item while preserving the current workflow state.
    for row in outer_tbody.find_all("tr", recursive=False):
        if row.select_one("a[href*='/profil/spieler/']") is None:
            continue
        player_records.append(
            parse_player_row(row, team_name, url, column_map, validation_warnings)
        )

    displayed_squad_size = extract_squad_size(soup)
    print(f"{team_name}: {len(player_records)} players extracted")
    if displayed_squad_size is not None and displayed_squad_size != len(player_records):
        warning = (
            f"WARNING: {team_name} reports squad size {displayed_squad_size} "
            f"but only {len(player_records)} player rows were extracted."
        )
        validation_warnings.append(warning)
        print(warning)

    return player_records, displayed_squad_size


## 4. Scrape every configured team

One Chrome instance is reused for the complete run. Each club is isolated by its own exception handler, and the browser is closed by the outer finally block even if a failure occurs.


In [5]:
scrape_started_at = datetime.now().astimezone()
validation_warnings: list[str] = []
errors: list[dict[str, Any]] = []
all_player_records: list[dict[str, Any]] = []
successful_teams: list[str] = []
displayed_squad_sizes: dict[str, int | None] = {}
driver = None

# Handle expected failures with a clear, actionable message.
try:
    driver = uc.Chrome(version_main=150)
    driver.set_window_size(1920, 1080)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)

    # Process each available item while preserving the current workflow state.
    for team_number, (team_name, url) in enumerate(TEAM_URLS.items(), start=1):
        print(f"[{team_number}/{len(TEAM_URLS)}] Loading {team_name}")
        # Handle expected failures with a clear, actionable message.
        try:
            team_records, displayed_size = scrape_team(
                driver, team_name, url, validation_warnings
            )
        except Exception as exc:
            error_message = clean_text(str(exc)) or repr(exc)
            errors.append(
                {
                    "team": team_name,
                    "url": url,
                    "error_type": type(exc).__name__,
                    "error": error_message,
                }
            )
            print(f"ERROR: {team_name}: {type(exc).__name__}: {error_message}")
        else:
            all_player_records.extend(team_records)
            successful_teams.append(team_name)
            displayed_squad_sizes[team_name] = displayed_size

        if team_number < len(TEAM_URLS):
            delay_seconds = random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS)
            time.sleep(delay_seconds)
finally:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
            print("Chrome driver closed.")
        except Exception as shutdown_error:
            print(
                "Chrome driver shutdown warning: "
                f"{type(shutdown_error).__name__}: {shutdown_error}"
            )
        finally:
            driver = None

scrape_finished_at = datetime.now().astimezone()


[1/18] Loading Bayern Munich
Bayern Munich: 28 players extracted
[2/18] Loading RB Leipzig
RB Leipzig: 31 players extracted
[3/18] Loading Bayer Leverkusen
Bayer Leverkusen: 32 players extracted
[4/18] Loading Borussia Dortmund
Borussia Dortmund: 29 players extracted
[5/18] Loading VfB Stuttgart
VfB Stuttgart: 34 players extracted
[6/18] Loading Eintracht Frankfurt
Eintracht Frankfurt: 29 players extracted
[7/18] Loading TSG Hoffenheim
TSG Hoffenheim: 30 players extracted
[8/18] Loading SC Freiburg
SC Freiburg: 29 players extracted
[9/18] Loading Mainz 05
Mainz 05: 30 players extracted
[10/18] Loading FC Augsburg
FC Augsburg: 29 players extracted
[11/18] Loading Werder Bremen
Werder Bremen: 34 players extracted
[12/18] Loading Borussia Mönchengladbach
Borussia Mönchengladbach: 30 players extracted
[13/18] Loading 1. FC Köln
1. FC Köln: 30 players extracted
[14/18] Loading Union Berlin
Union Berlin: 31 players extracted
[15/18] Loading Hamburger SV
Hamburger SV: 28 players extracted
[16

## 5. Build, sort, and validate the DataFrames

The player table retains every extracted record. Duplicate candidates are reported but not removed. The club summary is calculated only from player-level market values.


In [6]:
# Validate the input before continuing with later processing.
if len(successful_teams) + len(errors) != len(TEAM_URLS):
    raise RuntimeError(
        "Internal team-count mismatch: successful plus failed teams does not equal "
        "the requested team count."
    )

df = pd.DataFrame(all_player_records, columns=PLAYER_COLUMNS)
# Process each available item while preserving the current workflow state.
for numeric_column in (
    "team_id",
    "season_id",
    "shirt_number",
    "player_id",
    "age",
    "market_value_eur",
):
    df[numeric_column] = pd.array(df[numeric_column], dtype="Int64")

df = df.sort_values(
    ["team", "market_value_eur", "player_name"],
    ascending=[True, False, True],
    na_position="last",
    kind="stable",
).reset_index(drop=True)

known_teams = set(TEAM_URLS)
unknown_record_teams = set(df["team"].dropna()) - known_teams
# Validate the input before continuing with later processing.
if unknown_record_teams:
    raise RuntimeError(f"Extracted records contain unknown teams: {unknown_record_teams}")

# Handle key for reuse in the workflow.
def duplicate_key(row: pd.Series) -> str:
    # Choose the appropriate path for the current data state.
    if pd.notna(row["player_id"]):
        identity = f"id:{int(row['player_id'])}"
    else:
        identity = f"name:{clean_text(row['player_name'])}"
    return f"{row['team']}|{identity}"


# Choose the appropriate path for the current data state.
if df.empty:
    duplicate_df = pd.DataFrame(columns=["duplicate_key", *PLAYER_COLUMNS])
else:
    duplicate_keys = df.apply(duplicate_key, axis=1)
    duplicate_mask = duplicate_keys.duplicated(keep=False)
    duplicate_df = df.loc[duplicate_mask].copy()
    duplicate_df.insert(0, "duplicate_key", duplicate_keys.loc[duplicate_mask])

# Choose the appropriate path for the current data state.
if duplicate_df.empty:
    print("Duplicate check: no duplicate player keys found.")
else:
    print(f"Duplicate check: {len(duplicate_df)} candidate rows found; none removed.")


# Handle number for reuse in the workflow.
def summary_number(value: Any) -> int | float | None:
    if pd.isna(value):
        return None
    numeric_value = float(value)
    return int(numeric_value) if numeric_value.is_integer() else numeric_value


summary_rows: list[dict[str, Any]] = []
# Process each available item while preserving the current workflow state.
for team_name in successful_teams:
    team_players = df.loc[df["team"] == team_name]
    market_values = team_players["market_value_eur"].dropna()
    summary_rows.append(
        {
            "team": team_name,
            "players_scraped": len(team_players),
            "players_with_market_value": len(market_values),
            "total_market_value_eur": summary_number(
                team_players["market_value_eur"].sum(min_count=1)
            ),
            "average_market_value_eur": summary_number(market_values.mean()),
            "median_market_value_eur": summary_number(market_values.median()),
            "highest_market_value_eur": summary_number(market_values.max()),
        }
    )

club_summary_df = pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS)
club_summary_df = club_summary_df.sort_values(
    "total_market_value_eur",
    ascending=False,
    na_position="last",
    kind="stable",
).reset_index(drop=True)

team_counts_df = (
    df.groupby("team", dropna=False)
    .size()
    .rename("players_scraped")
    .to_frame()
)
if successful_teams:
    team_counts_df = team_counts_df.reindex(successful_teams, fill_value=0)


Duplicate check: no duplicate player keys found.


## 6. Serialize JSON and CSV outputs

Both files share the local scrape-start timestamp. JSON retains full player records and real nationality arrays; CSV uses a pipe-delimited nationality field.


In [7]:
# Handle json safe for reuse in the workflow.
def to_json_safe(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_safe(item) for item in value]
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        numeric_value = float(value)
        return None if math.isnan(numeric_value) else numeric_value
    # Handle expected failures with a clear, actionable message.
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if hasattr(value, "item"):
        return to_json_safe(value.item())
    return value


records_by_team: dict[str, list[dict[str, Any]]] = {
    team_name: [] for team_name in TEAM_URLS
}
# Process each available item while preserving the current workflow state.
for player_record in df.to_dict(orient="records"):
    safe_record = to_json_safe(player_record)
    records_by_team[safe_record["team"]].append(safe_record)

filename_timestamp = scrape_started_at.strftime("%Y%m%d_%H%M%S")
output_stem = f"transfermarkt_bundesliga_squads_2026_{filename_timestamp}"
json_output_path = base_directory / f"{output_stem}.json"
csv_output_path = base_directory / f"{output_stem}.csv"

players_with_market_values = int(df["market_value_eur"].notna().sum())
json_output = {
    "metadata": {
        "source": "Transfermarkt",
        "source_base_url": TRANSFERMARKT_BASE_URL,
        "season_id": SEASON_ID,
        "captured_at": scrape_started_at.isoformat(timespec="seconds"),
        "capture_finished_at": scrape_finished_at.isoformat(timespec="seconds"),
        "number_of_teams": len(TEAM_URLS),
        "teams_successfully_scraped": len(successful_teams),
        "teams_failed": len(errors),
        "number_of_players": len(df),
        "players_with_market_value": players_with_market_values,
        "displayed_squad_sizes": displayed_squad_sizes,
        "team_urls": TEAM_URLS,
    },
    "teams": records_by_team,
    "validation_warnings": validation_warnings,
    "errors": errors,
}

# Use the resource only within this controlled scope.
with json_output_path.open("w", encoding="utf-8", newline="\n") as json_file:
    json.dump(to_json_safe(json_output), json_file, ensure_ascii=False, indent=2)
    json_file.write("\n")

csv_df = df.copy()
csv_df["nationalities"] = csv_df["nationalities"].map(
    lambda values: "|".join(values) if isinstance(values, list) else ""
)
csv_df.to_csv(
    csv_output_path,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
    na_rep="",
)


## 7. Final report

The final report shows the output locations, player preview and shape, counts by team, calculated club summary, duplicate candidates, validation warnings, and scraping errors.


In [8]:
print("\nTransfermarkt squad scrape complete.\n")
print(f"Squad output directory: {base_directory}")
print(f"Teams requested: {len(TEAM_URLS)}")
print(f"Teams successfully scraped: {len(successful_teams)}")
print(f"Teams failed: {len(errors)}")
print(f"Players extracted: {len(df)}")
print(f"Players with market values: {players_with_market_values}")
print(f"\nJSON:\n{json_output_path.resolve()}")
print(f"\nCSV:\n{csv_output_path.resolve()}")

print("\nFirst player rows:")
display(df.head())
print(f"DataFrame shape: {df.shape}")

print("\nPlayer count by team:")
display(team_counts_df)

print("\nClub market-value summary:")
display(club_summary_df)

print("\nDuplicate candidates:")
# Choose the appropriate path for the current data state.
if duplicate_df.empty:
    print("None")
else:
    display(duplicate_df)

print("\nValidation warnings:")
# Choose the appropriate path for the current data state.
if validation_warnings:
    display(pd.DataFrame({"warning": validation_warnings}))
else:
    print("None")

print("\nScraping errors:")
# Choose the appropriate path for the current data state.
if errors:
    display(pd.DataFrame(errors, columns=["team", "url", "error_type", "error"]))
else:
    print("None")



Transfermarkt squad scrape complete.

Squad output directory: C:\kickbase project\outputs\transfermarkt\squads
Teams requested: 18
Teams successfully scraped: 18
Teams failed: 0
Players extracted: 545
Players with market values: 528

JSON:
C:\kickbase project\outputs\transfermarkt\squads\transfermarkt_bundesliga_squads_2026_20260823_175610.json

CSV:
C:\kickbase project\outputs\transfermarkt\squads\transfermarkt_bundesliga_squads_2026_20260823_175610.csv

First player rows:


,team,team_id,season_id,shirt_number,player_name,player_id,position,age,nationalities,contract_until,market_value,market_value_eur,player_url
0,1. FC Köln,3,2026,10,Said El Mala,1168219,Left Winger,19,[Germany],30/06/2031,€45.00m,45000000,https://www.transfermarkt.com/said-el-mala/pro...
1,1. FC Köln,3,2026,22,Jahmai Simpson-Pusey,942497,Centre-Back,20,[England],30/06/2030,€9.00m,9000000,https://www.transfermarkt.com/jahmai-simpson-p...
2,1. FC Köln,3,2026,9,Ragnar Ache,416380,Centre-Forward,28,"[Germany, Ghana]",30/06/2029,€9.00m,9000000,https://www.transfermarkt.com/ragnar-ache/prof...
3,1. FC Köln,3,2026,27,Thijs Dallinga,538964,Centre-Forward,26,[Netherlands],30/06/2027,€9.00m,9000000,https://www.transfermarkt.com/thijs-dallinga/p...
4,1. FC Köln,3,2026,8,Ísak Jóhannesson,579565,Central Midfield,23,"[Iceland, England]",30/06/2030,€8.00m,8000000,https://www.transfermarkt.com/isak-johannesson...


DataFrame shape: (545, 13)

Player count by team:


,players_scraped
team,
Bayern Munich,28
RB Leipzig,31
Bayer Leverkusen,32
Borussia Dortmund,29
VfB Stuttgart,34
Eintracht Frankfurt,29
TSG Hoffenheim,30
SC Freiburg,29
Mainz 05,30



Club market-value summary:


,team,players_scraped,players_with_market_value,total_market_value_eur,average_market_value_eur,median_market_value_eur,highest_market_value_eur
0,Bayern Munich,28,28,1077500000,3.848214e+07,30000000,170000000
1,Borussia Dortmund,29,29,529450000,1.825690e+07,18000000,55000000
2,Bayer Leverkusen,32,30,513950000,1.713167e+07,18000000,45000000
3,RB Leipzig,31,30,481350000,1.604500e+07,15500000,50000000
4,VfB Stuttgart,34,34,423450000,1.245441e+07,10000000,45000000
5,Eintracht Frankfurt,29,27,341350000,1.264259e+07,10000000,45000000
6,TSG Hoffenheim,30,30,294950000,9.831667e+06,7250000,35000000
7,SC Freiburg,29,29,236550000,8.156897e+06,7000000,25000000
8,Mainz 05,30,29,175200000,6.041379e+06,2500000,50000000
9,FC Augsburg,29,28,160400000,5.728571e+06,3250000,22000000



Duplicate candidates:
None

Validation warnings:
None

Scraping errors:
None


## 8. Export players at or above a configurable team percentile

This additional export compares players only with teammates who have a numeric market value. Set the percentage in the small parameter cell below. Missing market values are excluded, while ties at the configured threshold are retained.


In [9]:
# Change only this value to select a different within-team market-value percentile.
# Examples: 90.0 selects the 90th percentile; 95.0 selects the 95th percentile.
TEAM_MARKET_VALUE_PERCENTILE = 80.0

# Validate the input before continuing with later processing.
if not isinstance(TEAM_MARKET_VALUE_PERCENTILE, (int, float)):
    raise TypeError("TEAM_MARKET_VALUE_PERCENTILE must be a number from 0 to 100.")
# Validate the input before continuing with later processing.
if not 0 < TEAM_MARKET_VALUE_PERCENTILE <= 100:
    raise ValueError("TEAM_MARKET_VALUE_PERCENTILE must be greater than 0 and at most 100.")


In [10]:
percentile_quantile = TEAM_MARKET_VALUE_PERCENTILE / 100
percentile_filename_label = (
    f"p{TEAM_MARKET_VALUE_PERCENTILE:g}".replace(".", "_")
)
threshold_column = "team_market_value_percentile_threshold_eur"
team_percentile_df = df.loc[df["market_value_eur"].notna()].copy()

# Choose the appropriate path for the current data state.
if team_percentile_df.empty:
    team_percentile_df[threshold_column] = pd.Series(
        dtype="Float64"
    )
    team_percentile_df["market_value_percentile_within_team"] = pd.Series(
        dtype="Float64"
    )
else:
    market_value_groups = team_percentile_df.groupby("team")["market_value_eur"]
    team_percentile_df[threshold_column] = (
        market_value_groups.transform(
            lambda values: values.astype("float64").quantile(percentile_quantile)
        ).round(2)
    )
    team_percentile_df["market_value_percentile_within_team"] = (
        market_value_groups.rank(method="average", pct=True).mul(100).round(2)
    )
    team_percentile_df = team_percentile_df.loc[
        team_percentile_df["market_value_eur"]
        >= team_percentile_df[threshold_column]
    ].copy()

team_percentile_df = team_percentile_df[
    [
        *PLAYER_COLUMNS,
        threshold_column,
        "market_value_percentile_within_team",
    ]
].sort_values(
    ["team", "market_value_eur", "player_name"],
    ascending=[True, False, True],
    kind="stable",
).reset_index(drop=True)

team_percentile_csv_path = (
    base_directory / f"{output_stem}_team_market_value_{percentile_filename_label}.csv"
)
team_percentile_csv_df = team_percentile_df.copy()
team_percentile_csv_df["nationalities"] = team_percentile_csv_df[
    "nationalities"
].map(lambda values: "|".join(values) if isinstance(values, list) else "")
team_percentile_csv_df.to_csv(
    team_percentile_csv_path,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
    na_rep="",
)

print("\nTeam market-value percentile export complete.")
print(f"Configured percentile: {TEAM_MARKET_VALUE_PERCENTILE:g}")
print(f"Players exported: {len(team_percentile_df)}")
print(f"CSV:\n{team_percentile_csv_path.resolve()}")
display(team_percentile_df)



Team market-value percentile export complete.
Configured percentile: 80
Players exported: 118
CSV:
C:\kickbase project\outputs\transfermarkt\squads\transfermarkt_bundesliga_squads_2026_20260823_175610_team_market_value_p80.csv


,team,team_id,season_id,shirt_number,player_name,player_id,position,age,nationalities,contract_until,market_value,market_value_eur,player_url,team_market_value_percentile_threshold_eur,market_value_percentile_within_team
0,1. FC Köln,3,2026,10,Said El Mala,1168219,Left Winger,19,[Germany],30/06/2031,€45.00m,45000000,https://www.transfermarkt.com/said-el-mala/pro...,6700000.0,100.0
1,1. FC Köln,3,2026,22,Jahmai Simpson-Pusey,942497,Centre-Back,20,[England],30/06/2030,€9.00m,9000000,https://www.transfermarkt.com/jahmai-simpson-p...,6700000.0,92.59
2,1. FC Köln,3,2026,9,Ragnar Ache,416380,Centre-Forward,28,"[Germany, Ghana]",30/06/2029,€9.00m,9000000,https://www.transfermarkt.com/ragnar-ache/prof...,6700000.0,92.59
3,1. FC Köln,3,2026,27,Thijs Dallinga,538964,Centre-Forward,26,[Netherlands],30/06/2027,€9.00m,9000000,https://www.transfermarkt.com/thijs-dallinga/p...,6700000.0,92.59
4,1. FC Köln,3,2026,8,Ísak Jóhannesson,579565,Central Midfield,23,"[Iceland, England]",30/06/2030,€8.00m,8000000,https://www.transfermarkt.com/isak-johannesson...,6700000.0,85.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,Werder Bremen,86,2026,7,Samuel Mbangula,654991,Left Winger,22,"[Belgium, DR Congo]",30/06/2030,€8.00m,8000000,https://www.transfermarkt.com/samuel-mbangula/...,5000000.0,90.91
114,Werder Bremen,86,2026,14,Senne Lynen,338668,Defensive Midfield,27,[Belgium],NaN,€8.00m,8000000,https://www.transfermarkt.com/senne-lynen/prof...,5000000.0,90.91
115,Werder Bremen,86,2026,18,Eren Dinkçi,645774,Right Winger,24,"[Türkiye, Germany]",30/06/2027,€6.00m,6000000,https://www.transfermarkt.com/eren-dinkci/prof...,5000000.0,84.85
116,Werder Bremen,86,2026,11,Justin Njinmah,596153,Right Winger,25,"[Germany, Nigeria]",NaN,€5.00m,5000000,https://www.transfermarkt.com/justin-njinmah/p...,5000000.0,80.3


In [11]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")


Pruned 0 expired timestamped output(s).
